## Evaluate integration models - No bactch for hvgs

In [ ]:
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

## Load Libraries

In [1]:
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc
import subprocess
from joblib import parallel_backend
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import hdf5plugin

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [2]:
# Directories
ma.create_directories(dir_path = str(here('data/integration/second_pass')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/tmp')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/embedding')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/files')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/plot')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/objects')))
ma.create_directories(dir_path = str(here('data/integration/second_pass/models')))

/work/islet_cartography_scrna/data/integration/second_pass Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/tmp Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/embedding Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/files Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/plot Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/objects Directory already exists!
/work/islet_cartography_scrna/data/integration/second_pass/models Directory already exists!


In [3]:
# Saving
base_dir = str(here('data/integration/second_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
emb_dir = os.path.join(base_dir, 'embedding') 
objects_dir = os.path.join(base_dir, 'objects') 

In [4]:
hvgs = [500, 1000, 2000]
methods = ["scvi", "sysvi", "scpoli"]
adata_dir = tmp_dir  # directory with preprocessed HVG files
file_dir = os.path.join(base_dir, 'files')  # directory to save embeddings
model_dir = os.path.join(base_dir, 'models')   # directory to save models
key_batch = ["technical_integration", "ic_id_donor_integrate"]   # adjust to your obs keys
latent_dims_list = [15]
embedding_dims_list = [5, 10, 20]
label_key = 'study_cell_annotation_harmonized' 

In [5]:
# plotting
sc.set_figure_params(figsize=(5, 5), frameon=False)
# save path
sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
# PAth to enviroments
py_analysis = '/work/islet_cartography_scrna/scrna_cartography_py_analysis'
scpoli = '/work/islet_cartography_scrna/scrna_cartography_scpoli'

## Load data

In [ ]:
adata =sc.read_h5ad(here('data/anndata/AB_combined.h5ad'))

In [ ]:
# Add technical variation
adata.obs['technical_integration'] = (
    adata.obs['cell_nuclei'].astype(str) + "_" + adata.obs['count_quantification'].astype(str)
)

# copy of raw counts
adata.raw = adata.copy()

# make na unknown, otherwise it will not work
adata.obs[label_key] = adata.obs[label_key].fillna('unknown')

In [ ]:
# # subset for test
# # number per group
# n_per_group = 100000 // adata.obs['ic_id_donor_integrate'].nunique()

# # sample indices
# sampled_obs = (
#     adata.obs
#     .groupby('ic_id_donor_integrate', group_keys=False)
#     .apply(lambda x: x.sample(n=min(len(x), n_per_group), random_state=0))
# )

# # subset AnnData
# adata= adata[sampled_obs.index, :].copy()

In [ ]:
for n in hvgs:
    
    ad = adata.copy()

    print(f'Finding {n} highly variable genes')
    sc.pp.highly_variable_genes(
        ad,
        n_top_genes=n,
        flavor="seurat_v3",
        layer="counts",
        batch_key = None,
        subset=True)
    
    # Extract hvgs
    hvg_list = ad.var[ad.var['highly_variable']].index.tolist()
    
    # Save hvgs 
    hvg_path = os.path.join(file_dir, f'hvgs_{n}.txt')
    
    with open(hvg_path, 'w') as f:
        for gene in hvg_list:
            f.write(gene + '\n')
    
    print(f'Saved highly variable genes to {hvg_path}')
    
    # Save adata object
    adata_file = os.path.join(objects_dir, f"adata_{n}_hvg.h5ad")
    ad.write(adata_file)

    print(f'Saved adata object to {adata_file}')
    
    # Save adata object for scpoli
    adata_file_scpoli = os.path.join(objects_dir, f"adata_scpoli_{n}_hvg.h5ad")
    del ad.uns["log1p"]
    ad.write(adata_file_scpoli)
    
    print(f'Saved adata object for scpoli to {adata_file_scpoli}')
    
    for latent_dims in latent_dims_list:
        
        key_save = f"{'_'.join(key_batch)}_latent{latent_dims}"

        print(f'Running {key_save}')
        print(f'Running no integration')
        # Run unintegrated (PCA)
        result_int = subprocess.run([
            "conda", "run", "-p", py_analysis, "python", "run_no_int.py",
            str(n), adata_file, model_dir, emb_dir, key_save, str(latent_dims)
        ], capture_output=True, text=True)
        print(result_int.stderr)

        for embedding_dims in embedding_dims_list:
            key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"

            print(f'Running {key_save}')
            
            print(f'Running ScVI')
            # Run scVI
            result_scvi = subprocess.run([
                "conda", "run", "-p", py_analysis, "python", "run_scvi.py",
                str(n), adata_file, model_dir, emb_dir, key_batch[0], key_batch[1], key_save,  str(latent_dims), str(embedding_dims)
            ], capture_output=True, text=True)
            print(result_scvi.stderr)
            
            
            print(f'Running SysVI')
            # Run SysVI
            result_sysvi = subprocess.run([
                "conda", "run", "-p", py_analysis, "python", "run_sysvi.py",
                str(n), adata_file, model_dir, emb_dir, key_batch[0], key_batch[1], key_save, str(latent_dims), str(embedding_dims)
            ], capture_output=True, text=True)
            print(result_sysvi.stderr)
            
            
            print(f'Running ScPoli')
            # Run scPoli
            result_scpoli = subprocess.run([
                "conda", "run", "-p", scpoli, "python", "run_scpoli.py",
                str(n), adata_file_scpoli, model_dir, emb_dir, ",".join(key_batch), key_save, str(latent_dims), str(embedding_dims)
            ], capture_output=True, text=True)
            print(result_scpoli.stderr)
            

## Add latent embedding

In [ ]:
# Make dictionary of adata objects
adata_dict = {}

# Make copies per n
for n in hvgs:
    adata_dict[n] = adata.copy()

# Add unintegrated embedding:
for n in hvgs:
    for latent_dims in latent_dims_list:
        key_save = f"{'_'.join(key_batch)}_latent{latent_dims}"
        df_path = os.path.join(emb_dir, f"unintegrated_latent_embd_{n}_{key_save}.csv")

        if not os.path.exists(df_path):
                    print(f'Missing embedding: {df_path}')
                    continue

        latent_df = pd.read_csv(df_path, index_col=0)
        latent_df = latent_df.loc[adata.obs_names]  # ensure correct cell order

        # Add latent embeddings to the copy for this n
        obsm_key = f"unintegrated_{n}_{key_save}"
        adata_dict[n].obsm[obsm_key] = latent_df.to_numpy()

In [ ]:
# add for each method
for method in methods:
    for n in hvgs:
        for latent_dims in latent_dims_list:
            for embedding_dims in embedding_dims_list:
                key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"
                df_path = os.path.join(emb_dir, f"{method}_latent_embd_{n}_{key_save}.csv")
                if not os.path.exists(df_path):
                    print(f'Missing embedding: {df_path}')
                    continue
                
                latent_df = pd.read_csv(df_path, index_col=0)
                latent_df = latent_df.loc[adata.obs_names]  # ensure correct cell order
                
                # Add latent embeddings to the copy for this n
                obsm_key = f"{method}_{n}_{key_save}"
                adata_dict[n].obsm[obsm_key] = latent_df.to_numpy()

In [ ]:
# save adata and replace with object with directory
for n in hvgs:
    dir = os.path.join(objects_dir, f'adata_{n}_integration.h5ad')
    adata_dict[n].write(
        dir,
        compression=hdf5plugin.FILTERS["zstd"],
        compression_opts=hdf5plugin.Zstd(clevel=5).filter_options)
    adata_dict[n] = dir

In [6]:
adata_dict = {}
for n in hvgs:
    dir = os.path.join(objects_dir, f'adata_{n}_integration.h5ad')
    adata_dict[n] = dir

In [7]:
adata_dict

{500: '/work/islet_cartography_scrna/data/integration/second_pass/objects/adata_500_integration.h5ad',
 1000: '/work/islet_cartography_scrna/data/integration/second_pass/objects/adata_1000_integration.h5ad',
 2000: '/work/islet_cartography_scrna/data/integration/second_pass/objects/adata_2000_integration.h5ad'}

## Generate UMAP

In [ ]:
for n in hvgs:
    print(f'Loading adata object from {adata_dict[n]}')
    
    ad = sc.read_h5ad(adata_dict[n])
    
    pdf_path = os.path.join(plot_dir, f'UMAP_{str(n)}_integration.pdf')
    print(f"Saving PDF to: {pdf_path}")
    
    with PdfPages(pdf_path) as pdf:
        latent_keys = list(ad.obsm.keys())
        print(f"Latent keys found: {latent_keys}")
        
        for key in latent_keys:
            print(f"Computing neighbors and UMAP for {key}...")
            
            # Compute neighbors using parallel threads
            with parallel_backend('threading', n_jobs=60):
                sc.pp.neighbors(ad, use_rep=key, key_added=f"{key}_neighbors")
            
            # Compute UMAP
            sc.tl.umap(ad, neighbors_key=f"{key}_neighbors")
            ad.obsm[f"X_{key}_umap"] = ad.obsm["X_umap"].copy()
            
            # Define order for plotting
            order = [
                "beta", "alpha", "alpha_beta", "delta", "gamma", "epsilon", "acinar", "ductal",
                "endothelial", "co_expression", "immune", "leukocyte", "mast", "mesenchymal",
                "mhc_class_ii", "schwann", "stellate", "endocrine", "cycling", "excluded"
            ]
            pp_order = order + list(np.unique(ad.obs[key_batch[0]].values))
            
            # Generate UMAP plot and save to PDF
            fig = sc.pl.umap(
                ad,
                ncols=2,
                size=2,
                wspace=1,
                groups=pp_order,
                color=["technical_integration", "study_cell_annotation_harmonized"],
                title=[f"UMAP {key} (Technical)", f"UMAP {key} (Annotation)"],
                show=False,         # Prevent interactive display
                return_fig=True     # Return matplotlib figure object
            )
            
            pdf.savefig(fig)
            plt.close(fig)  # Free memory after saving each figure

    # Overwrite the adata object with compression
    ad.write(
        adata_dict[n],
        compression=hdf5plugin.FILTERS["zstd"],
        compression_opts=hdf5plugin.Zstd(clevel=5).filter_options
    )
    print(f"Finished processing {n}, PDF saved to {pdf_path}\n")

In [14]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=FutureWarning,
        message="Argument `use_highly_variable` is deprecated.*"
    )
    
# Benchmark for technical integration
hvgs = [2000]
batch_tech = {}
for n in hvgs:
    print(f'Loading adata object from {adata_dict[n]}')
    
    ad = sc.read_h5ad(adata_dict[n])

    embedding = [x for x in ad.obsm.keys() if not x.startswith("X") and not x.endswith("umap")]

    # Add X_pca
    ad.obsm["X_pca"] = ad.obsm[f'unintegrated_{n}_technical_integration_ic_id_donor_integrate_latent15']
    
    print(f'Starting batch corrction benchmark')
    
    bm = Benchmarker(
    ad,
    batch_key=key_batch[0],
    label_key=label_key,
    bio_conservation_metrics=None,
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=embedding,
    n_jobs=60)

    bm.benchmark()

    batch_tech[n] = bm
    
    print(f'Finished batch corrction benchmark')

Loading adata object from /work/islet_cartography_scrna/data/integration/second_pass/objects/adata_2000_integration.h5ad
Starting batch corrction benchmark
computing PCA
    with n_comps=50


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


    finished (0:06:38)


Metrics:  40%|████      | 2/5 [04:44<05:53, 117.81s/it, Batch correction: kbet_per_label]

INFO     co_expression consists of a single batch or is too small. Skip.                                           
INFO     excluded consists of a single batch or is too small. Skip.                                                
INFO     immune consists of a single batch or is too small. Skip.                                                  
INFO     leukocyte consists of a single batch or is too small. Skip.                                               
INFO     mesenchymal consists of a single batch or is too small. Skip.                                             
INFO     mhc_class_ii consists of a single batch or is too small. Skip.                                            
INFO     schwann consists of a single batch or is too small. Skip.                                                 


...

INFO     co_expression consists of a single batch or is too small. Skip.                                           
INFO     excluded consists of a single batch or is too small. Skip.                                                
INFO     immune consists of a single batch or is too small. Skip.                                                  
INFO     leukocyte consists of a single batch or is too small. Skip.                                               
INFO     mesenchymal consists of a single batch or is too small. Skip.                                             
INFO     mhc_class_ii consists of a single batch or is too small. Skip.                                            
INFO     schwann consists of a single batch or is too small. Skip.                                                 



Metrics:  60%|██████    | 3/5 [35:44<30:35, 917.90s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:  90%|█████████ | 9/10 [3:09:38<28:14, 1694.43s/it]correction: pcr_comparison]
                                                                                         
Metrics:  40%|████      | 2/5 [03:58<04:56, 98.86s/it, Batch correction: kbet_per_label]

INFO     co_expression consists of a single batch or is too small. Skip.                                           
INFO     excluded consists of a single batch or is too small. Skip.                                                
INFO     immune consists of a single batch or is too small. Skip.                                                  
INFO     leukocyte consists of a single batch or is too small. Skip.                                               
INFO     mesenchymal consists of a single batch or is too small. Skip.                                             
INFO     mhc_class_ii consists of a single batch or is too small. Skip.                                            
INFO     schwann consists of a single batch or is too small. Skip.                                                 



Metrics:  60%|██████    | 3/5 [20:55<17:15, 517.98s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings: 100%|██████████| 10/10 [3:30:35<00:00, 1263.59s/it]orrection: pcr_comparison]

                                                                                         

Finished batch corrction benchmark


In [15]:
import zstandard as zstd
hvgs = [2000]
for n in hvgs:
    bm_path = os.path.join(objects_dir, f'benchmark_{n}_technical_integration.pkl.zst')

    cctx = zstd.ZstdCompressor(level=5)  # compression level: 1 (fast) → 22 (max)
    with open(bm_path, "wb") as f:
        with cctx.stream_writer(f) as compressor:
            pickle.dump(batch_tech[n], compressor, protocol=pickle.HIGHEST_PROTOCOL)